# 13 - Multi-PDF Streaming Chatbot

## Purpose

This notebook adds streaming chatbot output to the multi-PDF RAG assistant.

The chatbot searches across all uploaded PDFs, retrieves relevant chunks, sends the context to GPT, and streams the answer progressively.

Each response also shows the source PDF file, page number, and chunk ID used as supporting context.

## Input

This notebook loads the existing multi-PDF Chroma index from:

```text
/tmp/chroma_multi_pdf_v1

In [0]:
%pip install langchain langchain-community langchain-openai pypdf tiktoken chromadb

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import getpass

openai_api_key = getpass.getpass("Enter your OpenAI API key: ")

if not openai_api_key:
    raise ValueError("OpenAI API key was not entered.")

os.environ["OPENAI_API_KEY"] = openai_api_key

print("OpenAI API key loaded for this session")

In [0]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import TokenTextSplitter

pdf_folder_path = "/Volumes/workspace/365pdf/365pdfupload/multi_pdf_docs"

files_info = dbutils.fs.ls(pdf_folder_path)

pdf_files = [
    file_info.path.replace("dbfs:", "")
    for file_info in files_info
    if file_info.path.lower().endswith(".pdf")
]

print("Number of PDF files found:", len(pdf_files))

for pdf_file in pdf_files:
    print(pdf_file)

all_pdf_docs = []

for pdf_file in pdf_files:
    print("\nLoading PDF:")
    print(pdf_file)

    loader = PyPDFLoader(pdf_file)
    docs = loader.load()

    file_name = os.path.basename(pdf_file)

    for doc in docs:
        doc.metadata["source_file"] = file_name
        doc.metadata["source_path"] = pdf_file
        doc.metadata["page_number"] = doc.metadata.get("page")
        doc.metadata["document_type"] = "pdf"

    all_pdf_docs.extend(docs)

print("\nTotal PDF page documents loaded:", len(all_pdf_docs))

token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=500,
    chunk_overlap=50
)

multi_pdf_token_docs = token_splitter.split_documents(all_pdf_docs)

for i, doc in enumerate(multi_pdf_token_docs):
    doc.metadata["chunk_id"] = i

print("Token chunks created:", len(multi_pdf_token_docs))

In [0]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
import shutil
import os

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

multi_pdf_persist_directory = "/tmp/chroma_multi_pdf_streaming_temp"

if os.path.exists(multi_pdf_persist_directory):
    shutil.rmtree(multi_pdf_persist_directory)
    print("Removed existing temporary Chroma directory")

multi_pdf_vectorstore = Chroma.from_documents(
    documents=multi_pdf_token_docs,
    embedding=embedding,
    persist_directory=multi_pdf_persist_directory
)

print("Temporary multi-PDF Chroma index created")
print("Indexed chunks:", len(multi_pdf_token_docs))

In [0]:
multi_pdf_retriever = multi_pdf_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 12,
        "lambda_mult": 0.7
    }
)

test_docs = multi_pdf_retriever.invoke("What is Agentic RAG?")

print("Retrieved docs:", len(test_docs))

for i, doc in enumerate(test_docs, start=1):
    print(f"\nDocument {i}")
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page_number"))
    print("Chunk:", doc.metadata.get("chunk_id"))
    print(doc.page_content[:500])
    print("-" * 80)

In [0]:
def format_multi_pdf_context(docs):
    context_text = ""
    source_lines = []

    for i, doc in enumerate(docs, start=1):
        source_file = doc.metadata.get("source_file", "Unknown file")
        page_number = doc.metadata.get("page_number", "Unknown page")
        chunk_id = doc.metadata.get("chunk_id", "Unknown chunk")
        document_type = doc.metadata.get("document_type", "Unknown type")

        context_text += f"""
Document {i}
Source File: {source_file}
Page Number: {page_number}
Chunk ID: {chunk_id}
Document Type: {document_type}

Content:
{doc.page_content}

--------------------
"""

        source_lines.append(
            f"- Document {i}: {source_file}, page={page_number}, chunk_id={chunk_id}"
        )

    return context_text, source_lines

In [0]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

chat_streaming = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    streaming=True
)

multi_pdf_streaming_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a helpful multi-document question-answering assistant.

Answer the user's question using only the retrieved context from the uploaded PDF files.

Rules:
- Use only the provided retrieved context.
- Do not use outside knowledge.
- If the answer is not present in the retrieved context, say:
  "I could not find this information in the uploaded documents."
- If multiple documents are relevant, synthesize the answer across them.
- Keep the answer clear, accurate, and beginner-friendly.
- Do not invent sources, page numbers, or facts.
"""
    ),
    (
        "human",
        """
Retrieved context from uploaded documents:

{context}

User question:

{question}
"""
    )
])

print("Streaming GPT model and prompt ready")

In [0]:
def stream_multi_pdf_answer(question):
    docs = multi_pdf_retriever.invoke(question)

    if len(docs) == 0:
        print("I could not find relevant information in the uploaded documents.")
        return

    context_text, source_lines = format_multi_pdf_context(docs)

    messages = multi_pdf_streaming_prompt.invoke({
        "context": context_text,
        "question": question
    })

    print("Answer:")
    print()

    for chunk in chat_streaming.stream(messages):
        if chunk.content:
            print(chunk.content, end="", flush=True)

    print()
    print()
    print("Sources used:")
    for source in source_lines:
        print(source)

In [0]:
stream_multi_pdf_answer("What is Agentic RAG?")

In [0]:
stream_multi_pdf_answer("Compare agentic RAG and multimodal RAG.")

In [0]:
print("Multi-PDF Streaming Chatbot is ready.")
print("Ask a question about the uploaded PDFs.")
print("Type 'exit' to stop.")
print("-" * 80)

while True:
    question = input("You: ")

    if question.lower().strip() in ["exit", "quit", "stop"]:
        print("Chatbot: Goodbye.")
        break

    print()
    stream_multi_pdf_answer(question)
    print("-" * 80)